<a href="https://colab.research.google.com/github/GiladBoudman/Haifa-Eco-Pulse/blob/main/Nature_Bytes_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ChatBot**

In [1]:
!pip install -q sentence-transformers google-generativeai pypdf

import os
import glob
import logging
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, util
import google.generativeai as genai
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# Mute the background warnings from the PDF reader
logging.getLogger("pypdf").setLevel(logging.ERROR)

# Add your API Configuration
my_api_key = ""
genai.configure(api_key=my_api_key)
model = genai.GenerativeModel('gemini-3.5-flash')

local_dir = "/content/"

print("Scanning Colab for dropped PDF files and loading AI... (This takes a few seconds)")
corpus = []
source_mappings = []

file_paths = glob.glob(os.path.join(local_dir, "*.pdf"))

if not file_paths:
    raise FileNotFoundError("No .pdf files found! Please drag your Article PDFs into the Colab folder panel on the left.")

# Chunking function
def chunk_text(text, words_per_chunk=300):
    """Splits a massive string into smaller chunks to optimize vector embeddings."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), words_per_chunk):
        chunk = " ".join(words[i:i + words_per_chunk])
        chunks.append(chunk)
    return chunks

# Extract Text from PDFs and Chunk It
for local_path in file_paths:
    try:
        reader = PdfReader(local_path)
        content = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                content += text + "\n"

        content = content.strip()
        if content:
            file_name = os.path.basename(local_path)
            article_chunks = chunk_text(content, words_per_chunk=300)
            for chunk in article_chunks:
                corpus.append(chunk)
                source_mappings.append({"title": file_name, "text": chunk})

    except Exception as e:
        pass

# Vector Database Setup
retriever = SentenceTransformer('all-MiniLM-L6-v2')
corpus_embeddings = retriever.encode(corpus, convert_to_tensor=True)

# Wipe screen clean before dashboard appearance
clear_output()

# Build the UI
search_input = widgets.Text(
    placeholder='Ask a question about ecology, Haifa, or statistics...',
    description='Query:',
    layout=widgets.Layout(width='600px')
)
search_button = widgets.Button(
    description='Search',
    button_style='info',
    icon='search'
)
output_area = widgets.Output(layout=widgets.Layout(border='1px solid #D6D3C4', padding='10px', margin='10px 0', background_color='#f9f9f9'))

def run_gemini_rag(b):
    # UI loading state
    search_button.disabled = True
    search_button.description = 'Analyzing...'
    search_button.button_style = 'warning'
    search_button.icon = 'spinner'

    with output_area:
        clear_output(wait=True)
        query = search_input.value

        if not query:
            # Reset UI if input is empty
            search_button.disabled = False
            search_button.description = 'Search'
            search_button.button_style = 'info'
            search_button.icon = 'search'
            return

        # Display a loading status message
        display(Markdown("⏳ **Haifa Eco Pulse Engine Active**\n*Extracting vector contexts and computing thermodynamic systems relationships. Please hold...*"))

        try:
            # Vector database match
            query_embedding = retriever.encode(query, convert_to_tensor=True)
            hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=5)[0]

            if hits[0]['score'] < 0.1:
                clear_output()
                print("I can only answer questions related to our research on air pollution, NO2, vegetation in Haifa, or statistical methods like Pearson and ANOVA.")
                return

            combined_context = ""
            source_titles = []
            for hit in hits:
                doc = source_mappings[hit['corpus_id']]
                combined_context += f"Context Block:\n{doc['text']}\n\n"
                source_titles.append(doc['title'])

            prompt = f"""
            You are the specialized core AI engine for the "Haifa Eco Pulse" platform, built for an advanced Ecological Models laboratory team.
            Your primary objective is to analyze environmental remote sensing data and research papers through Howard Odum's Systems Ecology framework (ecosystems as dynamic thermodynamic circuits).

            PROJECT CORE CONTEXT:
            - Study Area: Haifa, Israel (focusing on the complex interface between industrial zones like Haifa Bay refineries, urban transport networks, and localized topography/greenery).
            - System Inputs: Solar radiation (driving photosynthesis) and Anthropogenic emissions (NO2, air pollution from factories and vehicle traffic).
            - System Processes: Photosynthetic activity, the limiting impact of air pollutants on urban canopy health, and statistical interactions (Pearson correlation, ANOVA testing) analyzing the relationships between Nitrogen Dioxide (NO2), Land Surface Temperature (LST), and Normalized Difference Vegetation Index (NDVI).
            - System Outputs: Oxygen production, thermodynamic heat loss, and environmental feedback loops (such as rainfall scavenging/absorbing atmospheric NO2 gas).

            CRITICAL INSTRUCTIONS:
            1. Analyze the context blocks through this Systems Ecology lens whenever applicable, linking chemical and spatial data to inputs, processes, or outputs.
            2. Synthesize findings smoothly across the different text chunks to provide a comprehensive, academically sound answer.
            3. Explain complex ecological interactions and statistical methods in clean, accessible, and highly clear language.
            4. STRICT FORMULATION RULE: Do NOT use LaTeX or symbol wrapping ($ or $$) under any circumstance. Always write chemical configurations, indices, or values as plain structural text (e.g., write NO2, PM10, SO2, NDVI, Pearson r, and ANOVA).
            5. Ground every conclusion in the provided literature context blocks. If a connection can be logically derived using the provided data, do so transparently, but do not manufacture raw statistics or unsourced localized facts.

            Context Blocks:
            {combined_context}

            User Research Query: {query}
            """

            # Call Gemini API
            response = model.generate_content(prompt)

            clear_output()

            # Format the output
            markdown_text = f"### Analysis & Results\n{response.text}\n\n---\n**Sourced dynamically from project documentation & literature:**\n"
            for title in list(set(source_titles)):
                markdown_text += f"* {title}\n"

            display(Markdown(markdown_text))

        except Exception as e:
            clear_output()
            error_message = str(e)

            # Catch all variations of the 429 Quota error (rate limit)
            if "429" in error_message or "Quota" in error_message or "ResourceExhausted" in error_message:
                print("⏳ System Alert: API rate limit reached.")
                print("The free tier requires a brief cooldown. Please wait 30 seconds before submitting another search.")
            else:
                print(f"An unexpected error occurred: {e}")

        finally:
            # Reset UI
            search_button.disabled = False
            search_button.description = 'Search'
            search_button.button_style = 'info'
            search_button.icon = 'search'

search_button.on_click(run_gemini_rag)

rag_ui = widgets.VBox([
    widgets.HTML("<div style='font-size: 18px; font-weight: bold; margin-bottom: 10px;'>Haifa Eco Pulse Academic Assistant</div>"),
    widgets.HBox([search_input, search_button]),
    output_area
], layout=widgets.Layout(padding='10px'))

# Display the dashboard
display(rag_ui)